# Condition estimation with hydraulic sensor data

Estimate cycle-average cooling efficiency from pressure measurements in HydSys.
Each row represents one 60-second test cycle. The target is the continuous CE
virtual sensor, rather than the discrete cooler-condition labels in profile.txt.

## Learning goals

- summarize pressure waveforms into cycle-level features;
- align inputs and targets by cycle, despite different sampling rates;
- evaluate a regression baseline on later cycles.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdmdata").is_dir())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import pdmdata
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
pdmdata.download("hydsys")
pressure = pdmdata.load("hydsys", sensor="PS1").to_numpy()
efficiency = pdmdata.load("hydsys", sensor="CE").to_numpy()
assert pressure.shape[0] == efficiency.shape[0], "Cycle counts must match"
features = np.column_stack([
    pressure.mean(axis=1), pressure.std(axis=1),
    pressure.min(axis=1), pressure.max(axis=1),
])
target = efficiency.mean(axis=1)
features.shape, target.shape

In [ ]:
split = int(len(target) * 0.8)
model = ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1)
model.fit(features[:split], target[:split])
prediction = model.predict(features[split:])
actual = target[split:]
print(f"Cooling-efficiency MAE: {mean_absolute_error(actual, prediction):.3f}")

In [ ]:
plt.scatter(actual, prediction, s=8, alpha=0.4)
limits = [min(actual.min(), prediction.min()), max(actual.max(), prediction.max())]
plt.plot(limits, limits, "--", color="black")
plt.xlabel("Actual cycle-average cooling efficiency (%)")
plt.ylabel("Predicted cooling efficiency (%)")
plt.show()

## Interpretation and limitations

This baseline estimates a contemporaneous virtual sensor from pressure; it does
not predict future failures or remaining useful life. CE is used only as a
target. The split preserves cycle order, but repeated test-rig conditions may
still occur in both partitions. It does not establish generalization to other
machines or unseen fault conditions.